## Random Forest Model

Tree-based models are great when you want to know which features have the most influence on predictions in complex, nonlinear datasets

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.tree import plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, f1_score, recall_score
from sklearn.model_selection import GridSearchCV
from pickle import dump

In [2]:
train_data = pd.read_csv("data/clean/clean_train.csv")
test_data = pd.read_csv("data/clean/clean_test.csv")

x_train = train_data.drop(["diagnosis_M"], axis = 1)
y_train = train_data["diagnosis_M"]
x_test = test_data.drop(["diagnosis_M"], axis = 1)
y_test = test_data["diagnosis_M"]

x_train.head(10)

,id,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,symmetry_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
0,859471,9.029,17.33,58.79,250.5,0.10660,0.14130,0.31300,0.04375,0.2111,...,10.31,22.65,65.50,324.7,0.14820,0.43650,1.25200,0.17500,0.4228,0.11750
1,873593,21.090,26.57,142.70,1311.0,0.11410,0.28320,0.24870,0.14960,0.2395,...,26.68,33.48,176.50,2089.0,0.14910,0.75840,0.67800,0.29030,0.4098,0.12840
2,859196,9.173,13.86,59.20,260.9,0.07721,0.08751,0.05988,0.02180,0.2341,...,10.01,19.23,65.59,310.1,0.09836,0.16780,0.13970,0.05087,0.3282,0.08490
3,88466802,10.650,25.22,68.01,347.0,0.09657,0.07234,0.02379,0.01615,0.1897,...,12.25,35.19,77.98,455.7,0.14990,0.13980,0.11250,0.06136,0.3409,0.08147
4,858970,10.170,14.88,64.55,311.9,0.11340,0.08061,0.01084,0.01290,0.2743,...,11.02,17.45,69.86,368.6,0.12750,0.09866,0.02168,0.02579,0.3557,0.08020
5,84799002,14.540,27.54,96.73,658.8,0.11390,0.15950,0.16390,0.07364,0.2303,...,17.46,37.13,124.10,943.2,0.16780,0.65770,0.70260,0.17120,0.4218,0.13410
6,89143602,14.410,19.73,96.03,651.0,0.08757,0.16760,0.13620,0.06602,0.1714,...,15.77,22.13,101.70,767.3,0.09983,0.24720,0.22200,0.10210,0.2272,0.08799
7,868682,11.430,15.39,73.06,399.8,0.09639,0.06889,0.03503,0.02875,0.1734,...,12.32,22.02,79.93,462.0,0.11900,0.16480,0.13990,0.08476,0.2676,0.06765
8,8711003,12.250,17.94,78.27,460.3,0.08654,0.06679,0.03885,0.02331,0.1970,...,13.59,25.22,86.60,564.2,0.12170,0.17880,0.19430,0.08211,0.3113,0.08132
9,916838,19.890,20.26,130.50,1214.0,0.10370,0.13100,0.14110,0.09431,0.1802,...,23.73,25.23,160.50,1646.0,0.14170,0.33090,0.41850,0.16130,0.2549,0.09136


In [ ]:
#TRAIN

model = RandomForestClassifier(random_state=42)
model.fit(x_train, y_train)

RandomForestClassifier(random_state=42)

In [4]:
#PREDICT

y_pred = model.predict(x_test)

accuracy = accuracy_score(y_test, y_pred)

print(f'Accuracy: {accuracy:.2f}')

report = classification_report(y_test, y_pred)
print(report)


Accuracy: 0.96
              precision    recall  f1-score   support

         0.0       0.96      0.99      0.97        71
         1.0       0.98      0.93      0.95        43

    accuracy                           0.96       114
   macro avg       0.97      0.96      0.96       114
weighted avg       0.97      0.96      0.96       114



The default model provides the desired outcome, with precision above 95% and higher accuracy in detecting malignant tumors

## Optimization

In [5]:
importances = model.feature_importances_

sorted_idx = importances.argsort()[::-1]
for i in sorted_idx:
    print(f"Feature: {x_train.columns[i]}, Importance: {importances[i]}")

Feature: perimeter_worst, Importance: 0.13245220171545263
Feature: area_worst, Importance: 0.12541417636388535
Feature: concave points_worst, Importance: 0.10819714409823475
Feature: radius_worst, Importance: 0.09797170076559213
Feature: concave points_mean, Importance: 0.08688498882946928
Feature: concavity_worst, Importance: 0.052982513219583846
Feature: concavity_mean, Importance: 0.050980217464886714
Feature: perimeter_mean, Importance: 0.05084280644969702
Feature: area_mean, Importance: 0.04056064146488961
Feature: area_se, Importance: 0.03425835402883133
Feature: radius_mean, Importance: 0.026504624627295147
Feature: compactness_worst, Importance: 0.019491641102294656
Feature: compactness_mean, Importance: 0.018097619296440687
Feature: perimeter_se, Importance: 0.017197336007647945
Feature: texture_worst, Importance: 0.0154769378568013
Feature: symmetry_worst, Importance: 0.014336720771137547
Feature: radius_se, Importance: 0.013504378229687367
Feature: texture_mean, Importance: 

In [6]:
def warn(*args, **kwargs):
    pass
import warnings
warnings.warn = warn

from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators': [25, 50, 100, 200],
    'max_depth': [5, 10, 20, 30],
    'min_samples_split': [5, 10, 15],
    'min_samples_leaf': [1, 2, 3, 4],
    'max_features': ['auto', 'sqrt'],
    'bootstrap': [True, False]
}

grid_search = GridSearchCV(estimator=model, param_grid=param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid_search.fit(x_train, y_train)

print(f"Best parameters: {grid_search.best_params_}")
print(f"Best cross-validated score: {grid_search.best_score_}")

Best parameters: {'bootstrap': False, 'max_depth': 5, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 25}
Best cross-validated score: 0.9626373626373625


In [7]:
model = RandomForestClassifier(n_estimators= grid_search.best_params_['n_estimators'], max_depth = grid_search.best_params_['max_depth'],\
                                min_samples_leaf = grid_search.best_params_['min_samples_leaf'],\
                                      min_samples_split = grid_search.best_params_['min_samples_split'],\
                                            bootstrap= grid_search.best_params_['bootstrap'],\
                                                max_features= grid_search.best_params_['max_features'], random_state = 42)
model.fit(x_train, y_train)

RandomForestClassifier(bootstrap=False, max_depth=5, min_samples_split=5,
                       n_estimators=25, random_state=42)

In [8]:
y_pred = model.predict(x_test)

accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average='weighted')
recall = recall_score(y_test, y_pred, average='weighted')

print(f'Accuracy: {accuracy:.2f}')
print(f'F1 Score: {f1:.2f}')
print(f'Recall: {recall:.2f}')

report = classification_report(y_test, y_pred)
print(report)

Accuracy: 0.96
F1 Score: 0.96
Recall: 0.96
              precision    recall  f1-score   support

         0.0       0.96      0.99      0.97        71
         1.0       0.98      0.93      0.95        43

    accuracy                           0.96       114
   macro avg       0.97      0.96      0.96       114
weighted avg       0.97      0.96      0.96       114



In [9]:
best_hyperparams_string = f"_n_estimators-{grid_search.best_params_['n_estimators']}_max_depth-{grid_search.best_params_['max_depth']}_bootstrap-{grid_search.best_params_['bootstrap']}_max_features-{grid_search.best_params_['max_features']}"


dump(model, open(f"models/random_forest_classifier_{best_hyperparams_string}_42.sav", "wb"))